# Setup and Import

In [1]:
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
from osgeo import gdal
from scipy.ndimage import label, generic_filter

# Globals

## Constants and paths

Configuration for this step (step 6: compute size statistics for the vegetation
clumps produced by the previous step): which municipality (`procom`), index,
flight year and thresholding `method` to use, and the NoData convention on the
input clump mask (`255`, from step 5). Input/output paths are under
`output/<procom>/`.

In [2]:
start_t = time.time()

mask_nodata_value = 255    # NoData value used by step 5 for the clump mask
pxl_area = 0.04 # in m^2; pixel size = 20 cm = 0.2 m
index = 'NDVI_red'  # NDVI_red; ENDVI; NDVI_blue
procom = "059033"
method = 'Australian' # ['KDE', 'KMeans', 'KMedians', 'Australian']
city_dict = {'039014': 'Ravenna', 
             '065116': 'Salerno', 
             '031007': 'Gorizia', 
             '038008': 'Ferrara', 
             '051002': 'Arezzo', 
             '048017': 'Firenze',
             '010025': 'Genova',
             '037006': 'Bologna',
             '032006': 'Trieste',
             '015146': 'Milano',
             '063049': 'Napoli',
             '082053': 'Palermo',
             '001272': 'Torino',
             '092009': 'Cagliari',
             '058091': 'Roma',
             '027042': 'Venezia',
             '080063': 'Reggio Calabria',
             '087015': 'Catania',
             '083048': 'Messina',
             '059033': 'Ventotene'}
city_name = city_dict[procom]
flight_year = '2022'

# Input file produced by step 5 (threshold + clump-size filtering), e.g.:
# C:\Users\UTENTE\Downloads\JOS_areeverdi\output\059033\Ventotene_d1-059033-NDVI_red-2022-Australian_clump.tif
clump_file_name = city_name + '-' + procom + '-' + index + '-' + flight_year + '-' + method + '_clump.tif'

# Output stats CSV for this step
clump_stats_file_name = city_name + '-' + procom + '-' + index + '-' + flight_year + '-' + method + '--clump_stats.csv'

# All inputs/outputs for this procom live under output/<procom>/
base_path = "C:/Users/UTENTE/Downloads/JOS_areeverdi/"
output_path = base_path + "output/" + procom + "/"

print('output_path: ', output_path)

output_path:  C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/


In [3]:
print(os.path.join(output_path, clump_file_name))
print(os.path.join(output_path, clump_stats_file_name))

C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022-Australian_clump.tif
C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022-Australian--clump_stats.csv


In [4]:
print(index)
print(procom)
print(city_name)
print(flight_year)
print(method)

NDVI_red
059033
Ventotene_d1
2022
Australian


## Functions

I/O and statistics helpers used throughout the notebook:

- `read_geotiff_as_ndarray_with_attributes`: open a GeoTIFF with GDAL and return the array together with its projection/geotransform.
- `label_2d_binary_mask`: label connected clumps of vegetation pixels (4-neighborhood connectivity) via `scipy.ndimage.label`. NoData pixels are excluded before labeling (see the "Statistics" section below), so they are never counted as clumps.
- `percentile_freq` / `calculate_statistics`: compute a weighted percentile / a full statistics summary from a set of (value, frequency) pairs.
- `crea_grafo` / `tile_clumping` / `clumps_statistics_tiling`: split the (potentially huge) clump mask into horizontal strips, label clumps independently in each strip, then use a graph (`networkx`) to merge clumps that were artificially cut by a strip boundary, and finally compute size statistics (in pixels and m²) over the reconnected clumps.

In [5]:

def read_geotiff_as_ndarray_with_attributes(filename):
    ds = gdal.Open(filename)
    band = ds.GetRasterBand(1)
    arr = band.ReadAsArray()
    return arr, ds.GetProjection(), ds.GetGeoTransform()

def label_2d_binary_mask(image_masked):
    # Pixel connectivity: 4-neighborhood ("cross" pattern) -- two vegetation
    # pixels are considered part of the same clump only if they touch on an
    # edge (not diagonally).
    #
    # Note: image_masked must contain only 0 (non-vegetation/background) and
    # 1 (vegetation) -- any NoData pixels must already have been zeroed out by
    # the caller, otherwise scipy.ndimage.label would treat them as another
    # foreground value and incorrectly count them as clumps.
    connectivity_array = np.array([ [0,1,0],
                                    [1,1,1],
                                    [0,1,0]  ], dtype=np.int8)
    
    # label() assigns a unique id to every group of connected pixels ("clumps")
    # according to connectivity_array.
    labeled_array, num_features = label(image_masked, structure= connectivity_array)
    # print(labeled_array, '\n')
    return labeled_array, num_features
"""
def label_2d_binary_mask__prove(image_masked):
    # (experimental variant, kept commented out as in the original notebook)
    connectivity_array = np.array([ [0,1,0],
                                    [1,1,1],
                                    [0,1,0]  ])
    
    labeled_array = label(image_masked, structure= connectivity_array, output=image_masked)
    # print(labeled_array, '\n')
    return labeled_array

def remove_isolated_pixels_old(image):
  image = np.pad(image,1)
  s = np.zeros((3,3), dtype=int)
  s[0,1] = 1
  s[2,1] = 1
  s[1,2] = 1
  s[2,1] = 1
  s[1,0] = 1
  print("Pixels not connected will be removed from image:")
  print(s)
  for i in range(image.shape[0]-2):
      for j in range(image.shape[1]-2):
          ma = image[i:i+3, j:j+3]
          c = np.logical_and(ma, s)
          #d = np.any(c) & (ma[1,1]==1)
          #k = d*1
          if ma[1,1]==1:
             k = np.any(c)*1
             image[i+1, j+1] = k
      if i % 1000 == 0:
        p=(i+1)/image.shape[0]*100
        print("Progress: {:.2f}%".format(p))
  return image[1:-1,1:-1]

def custom_filter(image):
    #print(image.shape)
    s3 = np.zeros((3,3), dtype=np.int8)
    s3[0,1] = 1
    s3[2,1] = 1
    s3[1,2] = 1
    s3[2,1] = 1
    s3[1,0] = 1
    s1 = np.zeros((3,3), dtype=np.int8)
    s1[1,1] = 1

    c = np.logical_and(image, s3.flatten()) # pxl is isolated
    e = np.logical_and(image, s1.flatten()) # pxl = 1
    d = np.any(c)*np.any(e)*1
    return d

def remove_isolated_pixels(image):
  image = np.pad(image,1).astype(np.int8)
  image = generic_filter(image, custom_filter, [3,3])

  return image[1:-1,1:-1].astype(np.int8)

def clump_to_stats(clump_array):
  # see: https://stackoverflow.com/questions/65405390/how-to-use-np-unique-on-big-arrays
  clump_stats = np.array([[0,0]], dtype=np.int32)
  loc_values, loc_counts = np.unique(clump_array, return_counts=True) # original from web
  for idx in range(loc_values.shape[0]):
      np.append(clump_stats, [[loc_values, loc_counts[idx]]])

  return clump_stats[1:]

def clump_to_mask(clump_array):
  # see: https://stackoverflow.com/questions/65405390/how-to-use-np-unique-on-big-arrays
  clump_bin_array = np.zeros(clump_array.shape, dtype=np.int8)
  for i, loc in enumerate(ndimage.find_objects(clump_array)):
      loc_values, loc_counts = np.unique(clump_array[loc], return_counts=True) # original from web
      print(loc_values)
      for idx in range(loc_values.shape[0]):
          if loc_counts[idx]>min_pxl_num:
              clump_bin_array[loc] = np.logical_or(clump_bin_array[loc], np.where(clump_array[loc]==loc_values[idx], 1, 0))

  return clump_bin_array
"""

def percentile_freq(values, freqs, percentile):
    """
    Calculates the desired percentile for given values and their frequencies.
    
    Parameters:
    - values: np.array, values to calculate the percentile for
    - freqs: np.array, frequencies of the values
    - percentile: float, desired percentile (0 < percentile < 100)
    
    Returns:
    - float: the calculated percentile value
    """
    
    function_name = percentile_freq.__name__

    # Internal Check
    if not (0 < percentile < 100):
        error_msg = f"{function_name} error: Percentile must be an integer between 0 and 99, got {percentile}"
        raise ValueError(error_msg)
        
    if len(values) != len(freqs):
        error_msg = f"{function_name} error: Values and frequencies must have the same length"
        raise ValueError(error_msg)

    sorted_indices = np.argsort(values)
    sorted_values = np.array(values)[sorted_indices]
    sorted_freqs = np.array(freqs)[sorted_indices]
    
    # Find percentiles using the cumulative instead of expanding tha array, for efficiency 
    cumulative_freqs = np.cumsum(sorted_freqs)
    target_value = (percentile / 100.0) * cumulative_freqs[-1]
    percentile_index = np.searchsorted(cumulative_freqs, target_value)

    return sorted_values[percentile_index]

def calculate_statistics(values, freqs, city_name, statistics_column_name,  rounding=2):
    """
    Calculates main statistics for given values and their frequencies.
    
    Parameters:
    - values: np.array, values to calculate statistics for
    - freqs: np.array, frequencies of the values
    - city_name: str, name of the city for labeling purposes
    - statistics_column_name: str, name of the statistics column
    - rounding: int, rounding precision
    
    Returns:
    - pd.DataFrame: DataFrame with calculated statistics
    """
        
    # if rounding=0 then show as integer.
    if rounding == 0:
        fmt = int  # Convert to integer
    else:
        fmt = lambda x: round(x, rounding)  
    
    # Calculate statistics
    average = fmt(np.average(values, weights=freqs))
    sigma = fmt(np.sqrt(np.average((values - average) ** 2, weights=freqs)))
    
    first_quartile = fmt(percentile_freq(values, freqs, 25))
    median = fmt(percentile_freq(values, freqs, 50))
    third_quartile = fmt(percentile_freq(values, freqs, 75))
    ninety_fifth_percentile = fmt(percentile_freq(values, freqs, 95))
    max_value = fmt(np.max(values))
    min_value = fmt(np.min(values))
    
    column_1_name = f'Statistics {city_name}'   
    column_2_name = statistics_column_name
    
    # Creating the DataFrame with dynamic column names
    data = {
        column_1_name: ['Average', 'Sigma' , '1st Quartile', 'Median', '3rd Quartile', '95th Percentile', 'min', 'Max'],
        column_2_name: [average, sigma, first_quartile, median, third_quartile, ninety_fifth_percentile, min_value, max_value]
    }

    df_stats = pd.DataFrame(data)
    df_stats.set_index( column_1_name, inplace=True)

    return df_stats

def crea_grafo(tablecouple):
    # Builds an undirected graph where each node is a "local" clump id
    # (one per tile) and each edge connects two clump ids that were found
    # to be adjacent across a tile boundary. Each connected component of
    # this graph is therefore a single real-world clump that was split
    # across several tiles.
    G = nx.Graph()
    for li in range(len(tablecouple)):
        table = tablecouple[li].copy()
        G.add_edges_from((table[["upper","lower"]].values))
    components = list(nx.connected_components(G))
    return components, G

###############################################################################

def tile_clumping(clump_mask_array, mask_nodata_value=255, tile_h=200):
    '''
    Returns the size (in pixels) of every vegetation clump in clump_mask_array.

    The array is processed in horizontal strips of height `tile_h` (rather
    than labeled all at once) to bound peak memory usage on very large
    rasters. Since labeling each strip independently can artificially cut a
    clump that spans a strip boundary into two separate pieces, adjacent
    strips are compared at their shared border and any clumps found to touch
    are reconnected via a graph (see `crea_grafo`), then their pixel counts
    are summed back together.

    NoData pixels (`mask_nodata_value`) are treated as background: they are
    zeroed out before labeling, so they are never counted as part of a clump.
    '''
    
    Xdim = clump_mask_array.shape[0]
    i = 0
    
    n_tiles = (Xdim // tile_h)
    if Xdim % tile_h == 0:
        n_tiles -= 1
        
    print('number of tiles: ', n_tiles)

    # Foreground mask: vegetation pixels (1) only, with both background (0)
    # and NoData (mask_nodata_value) treated as non-clump background.
    veg_mask = np.where(clump_mask_array == 1, 1, 0).astype(np.int8)

    # First strip ("upper" diamond): label its clumps and record their sizes.
    # clump_upper is a DataFrame with each clump's id and pixel count.
    clump_array_upper, num_features_upper = label_2d_binary_mask(veg_mask[i:i+tile_h,:])
    clump_idx_upper, clump_counts_upper = np.unique(clump_array_upper, return_counts=True)
    clump_upper = pd.DataFrame({"clump_idx": clump_idx_upper[0:], "clump_counts": clump_counts_upper[0:]})
    clump_upper["clump_idx"] = clump_upper.clump_idx.apply(lambda x: str(i)+"_"+str(x))
    upper_border = clump_array_upper[-1,:]
    
    tablecouple = []
    clump_size_list = []

    for i in range(1, n_tiles+1):
        clump_size_list.append(clump_upper)
    
        # Next strip ("lower" diamond): label its clumps and record their sizes.
        clump_array_lower, num_features_lower = label_2d_binary_mask(veg_mask[i*tile_h:(i+1)*tile_h,:])
        clump_idx_lower, clump_counts_lower = np.unique(clump_array_lower, return_counts=True) 
        
        clump_lower = pd.DataFrame({"clump_idx": clump_idx_lower[0:], "clump_counts": clump_counts_lower[0:]})
        clump_lower["clump_idx"] = clump_lower.clump_idx.apply(lambda x: str(i)+"_"+str(x))
    
        # "lower_border" is the border row of the lower strip that touches
        # the upper strip.
        lower_border = clump_array_lower[0,:]
        
        match = np.logical_and(lower_border, upper_border)
        upper_match = upper_border[match]
        lower_match = lower_border[match]
    
        # Build a table of adjacent (upper, lower) clump id pairs at this
        # strip boundary.
        tablecouple_i = pd.DataFrame(np.unique(np.array([upper_match, lower_match]),axis=1).T, columns=["upper","lower"])
        tablecouple_i["upper"] = tablecouple_i.upper.apply(lambda x: str(i-1)+"_"+str(x))
        tablecouple_i["lower"] = tablecouple_i.lower.apply(lambda x: str(i)+"_"+str(x))
        tablecouple.append(tablecouple_i)
    
        clump_upper = clump_lower
        upper_border = clump_array_lower[-1,:]
    
    
    clump_size_list.append(clump_lower)
    clump_sizes = pd.concat(clump_size_list)
    
    comp, G = crea_grafo(tablecouple)
    
    # From the graph, build a DataFrame mapping each local clump id to the
    # merged (reconnected) clump id it belongs to.
    merged_clump_map = []
    for ci, connected_clump in enumerate(comp):
        for clump_id in list(connected_clump):
            merged_clump_map.append(["C_"+str(ci), clump_id])
    merged_clump_map = pd.DataFrame(merged_clump_map, columns=["idCC","clump_idx"])
    
    clump_sizes_aggregate = clump_sizes.merge(merged_clump_map, on="clump_idx", how="left")
    connected_only_mask = clump_sizes_aggregate.idCC.isna()==False
    clump_sizes_aggregate.loc[connected_only_mask, "clump_idx"] = clump_sizes_aggregate.loc[connected_only_mask, "idCC"]
    
    clump_sizes_aggregate = clump_sizes_aggregate[["clump_counts","clump_idx"]].groupby("clump_idx").sum()

    # Drop background/NoData clumps (id ending in "_0", i.e. label 0 within a
    # tile) that were not merged into a real clump, then sort largest first.
    clump_sizes_aggregate = clump_sizes_aggregate[(clump_sizes_aggregate.index.str.contains("C"))|(clump_sizes_aggregate.index.str.contains("_0")==False)].sort_values("clump_counts",ascending=False)
    
    return clump_sizes_aggregate
    
######################################################################
def clumps_statistics_tiling(mask_cleaned, mask_nodata_value=255, pxl_area=0.04, tile_h=5000):
    ''' Returns the main statistics of the vegetation clumps' pixel size (and their area in m^2). '''
    
    clumps_with_tiles = tile_clumping(mask_cleaned, mask_nodata_value=mask_nodata_value, tile_h=tile_h)
    # Statistics of clumps' size distribution
    clump_dimension_statistics = clumps_with_tiles.describe().round(1)
    clump_dimension_statistics = clump_dimension_statistics.iloc[1:]
    
    clump_dimension_statistics['clump_counts (m^2)'] = (clump_dimension_statistics['clump_counts']*pxl_area).round(1)
    clump_dimension_statistics.columns = ['clump_dimension (pixels)', 'clump_dimension (m^2)']

    return clump_dimension_statistics


# Data

#### Load clump mask file

Loads the final vegetation mask produced by step 5 (threshold + clump-size
filtering). Byte raster: `0` = non-vegetation, `1` = vegetation, `255` = NoData.

In [6]:
try:
    clump_mask_array, ds_proj, ds_geotransf = read_geotiff_as_ndarray_with_attributes(os.path.join(output_path, clump_file_name))
    print('Loaded GeoTiff file')
except Exception:
    clump_file_name += '.npy'
    clump_mask_array = np.load(os.path.join(output_path, clump_file_name))
    print('Loaded npy file')
print(clump_file_name)
print(clump_mask_array.shape)
print(clump_mask_array.dtype)

C:\Users\UTENTE\anaconda3\envs\geo\Lib\site-packages\osgeo\gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Loaded GeoTiff file
Ventotene_d1-059033-NDVI_red-2022-Australian_clump.tif
(6526, 5309)
uint8


# Statistics

Computes the size distribution (in pixels and m²) of the vegetation clumps in
the loaded mask:
- split the mask into horizontal strips and label connected clumps in each (see `tile_clumping`)
- reconnect clumps that were artificially cut by a strip boundary, using a graph of adjacent clump ids
- exclude NoData pixels from labeling, so they are never counted as clumps
- compute summary statistics (mean, std, quartiles, min/max) over the reconnected clump sizes

In [7]:
pd_frame = clumps_statistics_tiling(clump_mask_array, mask_nodata_value=mask_nodata_value, pxl_area=pxl_area, tile_h=100)

number of tiles:  65


In [8]:
pd_frame

,clump_dimension (pixels),clump_dimension (m^2)
mean,33728.9,1349.2
std,102677.7,4107.1
min,2651.0,106.0
25%,4230.2,169.2
50%,9667.5,386.7
75%,23514.2,940.6
max,688605.0,27544.2


In [9]:
end_t = time.time()
e_t = end_t-start_t
print('Total time: ', e_t)

Total time:  5.881860256195068


## Export

Writes the clump size statistics to `output_path` (`output/<procom>/`) as
`<city_name>-<procom>-<index>-<flight_year>-<method>--clump_stats.csv`
(`;`-separated, comma as decimal separator).

In [10]:
pd_frame.to_csv(os.path.join(output_path, clump_stats_file_name), sep=';', decimal=',')
print('CSV saved: ', os.path.join(output_path, clump_stats_file_name))

CSV saved:  C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022-Australian--clump_stats.csv
